# DataSphere Dataset Init — tmj

Запустить **один раз** для инициализации датасета `tmj`.

Скачивает NIfTI кропы (800 MB) + метки из GitHub Release `crops-v2` (heatmap detector v1).

> Если датасет `tmj` уже существует — удали его в UI DataSphere перед запуском.

In [ ]:
#!:bash
#pragma dataset init tmj --size 20Gb

set -e
DS=/home/jupyter/datasets/tmj

echo '=== Скачиваю кропы crops-v2 (800 MB) ==='
wget --show-progress -O $DS/detector_crops_v2.tar.gz \
  https://github.com/tzopiz/MasterProject/releases/download/crops-v2/detector_crops_v2.tar.gz

echo '=== Распаковываю ==='
tar -xzf $DS/detector_crops_v2.tar.gz -C $DS
rm $DS/detector_crops_v2.tar.gz

echo '=== Скачиваю метки ==='
wget -q -O $DS/tmj_position_labels.json \
  https://github.com/tzopiz/MasterProject/releases/download/crops-v1/tmj_position_labels.json
wget -q -O $DS/manifest_private.json \
  https://github.com/tzopiz/MasterProject/releases/download/crops-v1/manifest_private.json

echo '=== Готово ==='
ls $DS
find $DS/detector_crops_v2 -name '*.nii.gz' | wc -l

После успешной инициализации датасет будет доступен по пути `/home/jupyter/datasets/tmj/` (read-only).

Структура:
```
/home/jupyter/datasets/tmj/
  detector_crops_v2/study_0001/study_0001_left.nii.gz
  detector_crops_v2/study_0001/study_0001_right.nii.gz
  ...
  tmj_position_labels.json
  manifest_private.json
```

Далее запускай `train_binary_position_classifier.ipynb`.

In [ ]:
"""Проверка датасета — запускай после инициализации."""
import json, os
import nibabel as nib
import numpy as np
from pathlib import Path

DS = Path("/home/jupyter/datasets/tmj")
CROPS_DIR = DS / "detector_crops_v2"

print("=" * 55)
print("ПРОВЕРКА ДАТАСЕТА tmj")
print("=" * 55)

# ── 1. Файлы верхнего уровня ───────────────────────────────
print("\n── Файлы в датасете ──")
for p in sorted(DS.iterdir()):
    size = p.stat().st_size / 1e6 if p.is_file() else sum(
        f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
    print(f"  {p.name:<35s} {'DIR' if p.is_dir() else 'FILE':5s}  {size:.1f} MB")

# ── 2. Кропы ──────────────────────────────────────────────
all_nii = sorted(CROPS_DIR.rglob("*.nii.gz"))
studies  = sorted({p.parent.name for p in all_nii})
n_left   = len([p for p in all_nii if "_left"  in p.name])
n_right  = len([p for p in all_nii if "_right" in p.name])

print(f"\n── Кропы ({CROPS_DIR.name}) ──")
print(f"  Исследований: {len(studies)}")
print(f"  Левых кропов: {n_left}")
print(f"  Правых кропов: {n_right}")
print(f"  Всего .nii.gz: {len(all_nii)}")

missing_left  = [s for s in studies if not (CROPS_DIR/s/f"{s}_left.nii.gz").exists()]
missing_right = [s for s in studies if not (CROPS_DIR/s/f"{s}_right.nii.gz").exists()]
if missing_left or missing_right:
    print(f"  ⚠️  Нет left:  {missing_left}")
    print(f"  ⚠️  Нет right: {missing_right}")
else:
    print("  ✓ Все пары L/R на месте")

# ── 3. Форма и содержимое одного кропа ────────────────────
sample = all_nii[0]
img = nib.load(str(sample))
vol = np.asarray(img.dataobj, dtype=np.float32)
print(f"\n── Пример кропа: {sample.name} ──")
print(f"  Форма: {vol.shape}")
print(f"  Dtype: {vol.dtype}")
print(f"  Диапазон HU: [{vol.min():.0f}, {vol.max():.0f}]")
print(f"  Среднее: {vol.mean():.1f}")

# ── 4. Метки ──────────────────────────────────────────────
labels_path = DS / "tmj_position_labels.json"
manifest_path = DS / "manifest_private.json"
print(f"\n── Метки ──")
if labels_path.exists():
    labels = json.loads(labels_path.read_text())
    n_patients = len(labels.get("patients", []))
    print(f"  tmj_position_labels.json: {n_patients} пациентов ✓")
else:
    print("  ⚠️  tmj_position_labels.json не найден!")

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    n_studies = len(manifest.get("studies", []))
    print(f"  manifest_private.json:    {n_studies} исследований ✓")
else:
    print("  ⚠️  manifest_private.json не найден!")

print("\n" + "=" * 55)
ok = (len(studies) > 0 and not missing_left and not missing_right
      and labels_path.exists() and manifest_path.exists())
print("СТАТУС:", "✅ ВСЁ ГОТОВО" if ok else "❌ ЕСТЬ ПРОБЛЕМЫ")
print("=" * 55)